# Prediksi Tingkat Obesitas dari Kebiasaan Makan dan Gaya Hidup
**Kelompok 5 — STMIK Pontianak — Predictive Analytics**

Pertanyaan: dari kebiasaan makan dan gaya hidup sehari-hari, apakah seseorang masuk kelompok **Kurang, Normal, atau Lebih** (berat badan berlebih)?

Dataset: Obesity Prediction Dataset (Kaggle, ruchikakumbhar) — 2.111 orang dari Meksiko, Peru, dan Kolombia. Bukan data Indonesia, jadi hasilnya tentang pola kebiasaan manusia secara umum, bukan tentang orang Indonesia secara khusus.

Jalankan sel demi sel dari atas ke bawah (Runtime > Run all di Colab).

## 1. Muat data
Sumber CSV: file `obesity-2111.csv` di repo yang sama. Di Colab, sel di bawah membaca langsung dari GitHub.

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/awangsuryarmdhn/weight-loss-regression/main/obesity-2111.csv"
df = pd.read_csv(URL)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
print(df.columns.tolist())
print("Data kosong:", int(df.isna().sum().sum()))
df.head()

## 2. Kenali targetnya: 7 level
Target aslinya punya 7 level. Semuanya terisi cukup seimbang, tidak ada level yang mendominasi.

In [ ]:
print(df["Obesity"].value_counts())
print(df["Obesity"].value_counts(normalize=True).round(3))
print("\nBaseline 7 kelas (selalu tebak level terbanyak):",
      round(df["Obesity"].value_counts(normalize=True).max() * 100, 2), "%")

## 3. BMI: rumus yang menentukan label
BMI = berat (kg) / tinggi (m)². Rata-rata BMI tiap level di bawah ini menunjukkan labelnya hampir ditentukan oleh BMI — dan itu penting, karena nanti akan jadi jebakan.

In [ ]:
df["BMI"] = df["Weight"] / (df["Height"] ** 2)
print(df.groupby("Obesity")["BMI"].agg(["mean", "min", "max", "count"]).round(2))
print("\nBatas BMI yang umum: Kurang < 18,5 | Normal 18,5–24,9 | Overweight 25–29,9 | Obesitas >= 30")

## 4. Sederhanakan 7 level menjadi 3 kelas
7 level terlalu rinci untuk presentasi: beda Overweight I dan II, atau Obesitas I dan II, itu soal derajat, bukan soal "lebih atau normal". Jadi digabung menjadi 3 kelas: **Kurang, Normal, Lebih**. Kelas Lebih jadi besar (73,5%), jadi baseline-nya ikut naik — itu harus diingat saat membaca akurasi.

In [ ]:
def ke_3_kelas(x):
    if x == "Insufficient_Weight":
        return "Kurang"
    if x == "Normal_Weight":
        return "Normal"
    return "Lebih"

df["Kelas3"] = df["Obesity"].apply(ke_3_kelas)
print(df["Kelas3"].value_counts())
print(df["Kelas3"].value_counts(normalize=True).round(3))
print("\nBaseline 3 kelas (selalu tebak 'Lebih'):", round(df["Kelas3"].value_counts(normalize=True).max() * 100, 2), "%")

## 5. Lihat dulu kebiasaannya (EDA)
Sebelum memodel, lihat pola kasarnya: kelompok Lebih lebih tua, lebih jarang olahraga, dan jauh lebih sering punya riwayat keluarga overweight.

In [ ]:
print(df.groupby("Kelas3")[["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE"]].mean().round(2))
print("\nSering makan tinggi kalori (FAVC = ya):")
print(df.groupby("Kelas3")["FAVC"].apply(lambda s: (s == "yes").mean()).round(3))
print("\nPunya riwayat keluarga overweight:")
print(df.groupby("Kelas3")["family_history"].apply(lambda s: (s == "yes").mean()).round(3))

## 6. Fungsi latih model (dipakai untuk semua skenario)
Tiga model dibandingkan: Decision Tree (maksimal 6 tingkat, supaya masih bisa dijelaskan), Random Forest (200 pohon), dan Regresi Logistik. Data dibagi 80% latih / 20% uji, acak dengan seed 42.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

def latih(X, y, label):
    X_latih, X_uji, y_latih, y_uji = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    angka = X.select_dtypes(include=[np.number]).columns.tolist()
    kategori = [c for c in X.columns if c not in angka]
    pra = ColumnTransformer([
        ("angka", StandardScaler(), angka),
        ("kategori", OneHotEncoder(handle_unknown="ignore"), kategori)])
    hasil = {}
    for nama, model in [
        ("Decision Tree", DecisionTreeClassifier(random_state=42, max_depth=6)),
        ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ("Regresi Logistik", LogisticRegression(max_iter=1000)),
    ]:
        pipa = Pipeline([("pra", pra), ("model", model)])
        pipa.fit(X_latih, y_latih)
        akurasi = accuracy_score(y_uji, pipa.predict(X_uji))
        hasil[nama] = round(float(akurasi), 4)
        print(f"{label} — {nama}: {akurasi:.2%}")
    return hasil

## 7. Skenario A: 7 kelas, DENGAN Tinggi + Berat
Akurasi tinggi (sekitar 97%). Tapi hati-hati: ini belum tentu prestasi — lihat skenario berikutnya.

In [ ]:
X_penuh = df.drop(columns=["Obesity", "Kelas3"])
latih(X_penuh, df["Obesity"], "7 kelas, DENGAN Tinggi+Berat")

## 8. Skenario B: 7 kelas, TANPA Tinggi + Berat (hanya kebiasaan)
Akurasi jatuh. Random Forest masih 85,34% — jauh di atas baseline 16,63% — tapi Decision Tree dan Regresi Logistik turun banyak. Artinya: menebak level rinci (I vs II vs III) dari kebiasaan saja memang sulit.

In [ ]:
X_kebiasaan = df.drop(columns=["Obesity", "Kelas3", "Height", "Weight", "BMI"])
latih(X_kebiasaan, df["Obesity"], "7 kelas, TANPA Tinggi+Berat")

## 9. Skenario C: 3 kelas, DENGAN Tinggi + Berat — INI JEBAKANNYA
Akurasi 99,5%. Kedengarannya hebat, sampai kita sadar: labelnya memang ditentukan dari BMI, dan BMI dihitung dari Tinggi + Berat. Model ini cuma menghitung ulang BMI lewat jalan memutar. Akurasi 99% di sini bukan prestasi, tapi tanda kebocoran target.

In [ ]:
latih(X_penuh, df["Kelas3"], "3 kelas, DENGAN Tinggi+Berat")
print("\nBukti: model HANYA dari BMI saja:")
latih(df[["BMI"]], df["Kelas3"], "3 kelas, HANYA BMI")

## 10. Skenario D: 3 kelas, TANPA Tinggi + Berat — MODEL UTAMA
Ini model yang dipakai. Dari kebiasaan saja, Random Forest 90,31% — jelas di atas baseline 73,52%, dan tidak bocor lewat BMI. Inilah prediksi yang jujur dan berguna: mendeteksi risiko dari kebiasaan, sebelum timbangan naik.

In [ ]:
hasil_utama = latih(X_kebiasaan, df["Kelas3"], "3 kelas, TANPA Tinggi+Berat (MODEL UTAMA)")
hasil_utama

## 11. Faktor apa yang paling berpengaruh? (model utama)
Tidak ada satu faktor yang mendominasi — modelnya seimbang. Umur paling besar (13,7%), lalu jumlah makan utama, frekuensi olahraga, kebiasaan ngemil, air minum, dan waktu layar, masing-masing sekitar 7–8%.

In [ ]:
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_kebiasaan, df["Kelas3"], test_size=0.2, random_state=42, stratify=df["Kelas3"])
angka = X_kebiasaan.select_dtypes(include=[np.number]).columns.tolist()
kategori = [c for c in X_kebiasaan.columns if c not in angka]
pra = ColumnTransformer([
    ("angka", "passthrough", angka),
    ("kategori", OneHotEncoder(handle_unknown="ignore"), kategori)])
rf = Pipeline([("pra", pra), ("model", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))])
rf.fit(X_latih, y_latih)
nama_fitur = pra.fit(X_latih).get_feature_names_out()
penting = sorted(zip(nama_fitur, rf.named_steps["model"].feature_importances_), key=lambda t: -t[1])
for nama, nilai in penting[:12]:
    print(f"{nama}: {nilai:.3f}")

## 12. Demo: dua profil kebiasaan
Profil sehat: 22 tahun, tidak ada riwayat keluarga, jarang makanan tinggi kalori, sayur tiap makan, olahraga teratur, jalan kaki.
Profil berisiko: 28 tahun, ada riwayat keluarga, sering makanan tinggi kalori, jarang sayur, jarang olahraga, sering ngemil, banyak waktu layar.
Perhatikan: model mengeluarkan peluang, bukan vonis. Profil berisiko pun masih punya peluang Normal yang cukup besar.

In [ ]:
profil_sehat = {
    "Gender": "Female", "Age": 22, "family_history": "no", "FAVC": "no",
    "FCVC": 3.0, "NCP": 3.0, "CAEC": "no", "SMOKE": "no", "CH2O": 2.0,
    "SCC": "no", "FAF": 2.0, "TUE": 1.0, "CALC": "no", "MTRANS": "Walking"}
profil_berisiko = {
    "Gender": "Male", "Age": 28, "family_history": "yes", "FAVC": "yes",
    "FCVC": 1.0, "NCP": 2.0, "CAEC": "Frequently", "SMOKE": "no", "CH2O": 1.0,
    "SCC": "no", "FAF": 0.0, "TUE": 3.0, "CALC": "Sometimes", "MTRANS": "Automobile"}

for label, profil in [("Profil sehat", profil_sehat), ("Profil berisiko", profil_berisiko)]:
    baris = pd.DataFrame([profil])
    prediksi = rf.predict(baris)[0]
    peluang = dict(zip(rf.classes_, rf.predict_proba(baris)[0].round(3)))
    print(label, "->", prediksi, peluang)

## Kesimpulan
1. Dari kebiasaan saja, model bisa membedakan Kurang / Normal / Lebih dengan akurasi 90,31% (Random Forest) — jelas di atas baseline 73,52%.
2. Kalau Tinggi + Berat dimasukkan, akurasi jadi 99,5% — tapi itu jebakan: labelnya memang dihitung dari BMI. Model yang terlalu akurat patut dicurigai.
3. Tidak ada satu kebiasaan yang menentukan semuanya. Umur, jumlah makan, olahraga, ngemil, air minum, dan waktu layar masing-masing menyumbang sekitar 7–14%.
4. Ini prediksi risiko dari survei kebiasaan, bukan diagnosis medis. Datanya dari Meksiko, Peru, dan Kolombia — bukan Indonesia. Riwayat keluarga menaikkan risiko, tapi bukan takdir.

Sumber data: https://www.kaggle.com/datasets/ruchikakumbhar/obesity-prediction